In [1]:
import numpy as np
import pandas as pd

# Load from EDA output
df = pd.read_csv('data_raw.csv')
df['capturedAt'] = pd.to_datetime(df['capturedAt'])
print(f'Loaded: {df.shape}')

Loaded: (306226, 27)


# Preprocessing
Cleaning and tidying up the data before it is used by the model.
Preprocessing answers 3 questions:
1. How to handle missing values?
2. How to convert text to numbers?
3. How to convert dates into useful features?

## 1. Copy Original Data
Before changing anything, copy df to df2 so the original data remains safe. If something goes wrong, we don't need to reread the CSV from the beginning.


In [2]:
df2 = df.copy()
print(f"original df  : {df.shape}")
print(f"copied df2   : {df2.shape}")

original df  : (306226, 27)
copied df2   : (306226, 27)


## 2. Drop Useless Columns
stock and normal_stock are 98.8% empty, they must be dropped because there is no information the model can learn from columns that are almost entirely empty.

In [3]:
df2 = df2.drop(columns=['stock', 'normal_stock'])

print(f"Columns after drop: {df2.shape[1]}")
print(f"Remaining columns : {list(df2.columns)}")

Columns after drop: 25
Remaining columns : ['capturedAt', 'shopId', 'itemId', 'modelId', 'price', 'priceBeforeDiscount', 'promotionId', 'cat_id', 'raw_discount', 'show_discount', 'brand', 'is_free_shipping', 'is_pre_order', 'item_price_min', 'item_price_max', 'review_rating', 'total_rating_count', 'cmt_count', 'shop_rating', 'shop_response_rate', 'shop_follower_count', 'is_official_shop', 'is_verified', 'is_preferred_plus_seller', 'date']


## 3. Fill Missing Values
Each column has a different filling strategy depending on its context and meaning.

In [4]:
print("Missing values before filling:")
print(df2.isnull().sum()[df2.isnull().sum() > 0])
print("")
# brand → fill with 'unknown'
df2['brand'] = df2['brand'].fillna('unknown')

# shop_response_rate → fill with median
median_response = df2['shop_response_rate'].median()
df2['shop_response_rate'] = df2['shop_response_rate'].fillna(median_response)

# Check the results
print("Missing values after filling:")
print(df2.isnull().sum()[df2.isnull().sum() > 0])

Missing values before filling:
brand                 105664
shop_response_rate      1199
dtype: int64

Missing values after filling:
Series([], dtype: int64)


## 4. Convert Booleans from Text to Numbers
Columns is_free_shipping, is_pre_order, etc. contain 't' and 'f'.
ML models cannot read text, so they must be converted to 1 and 0.

In [5]:
bool_cols = ['is_free_shipping', 'is_pre_order',
             'is_official_shop', 'is_verified',
             'is_preferred_plus_seller']

print("Before boolean conversion:")
print(df2[bool_cols].head(), end="\n\n")

for col in bool_cols:
    df2[col] = df2[col].map({'t': 1, 'f': 0})


print("After boolean conversion:")
print(df2[bool_cols].head())


Before boolean conversion:
  is_free_shipping is_pre_order is_official_shop is_verified  \
0                f            f                f           f   
1                f            f                t           f   
2                f            f                t           f   
3                f            f                t           f   
4                f            f                t           f   

  is_preferred_plus_seller  
0                        f  
1                        f  
2                        f  
3                        f  
4                        f  

After boolean conversion:
   is_free_shipping  is_pre_order  is_official_shop  is_verified  \
0                 0             0                 0            0   
1                 0             0                 1            0   
2                 0             0                 1            0   
3                 0             0                 1            0   
4                 0             0              

## 5. Convert Date to Numeric Features
capturedAt is still in datetime text format. We extract the time components: month, day, hour, and day of the week. The model cannot process date text directly.

In [6]:
print(df2['capturedAt'].dtype)
print(df2['capturedAt'].head())

datetime64[ns]
0   2025-01-01 22:37:36.424
1   2025-01-02 01:45:48.634
2   2025-01-02 01:45:48.634
3   2025-01-02 01:45:48.634
4   2025-01-02 01:45:48.634
Name: capturedAt, dtype: datetime64[ns]


In [7]:
df2['capturedAt'] = pd.to_datetime(df2['capturedAt'])

df2['month']     = df2['capturedAt'].dt.month       # 1-12
df2['day']       = df2['capturedAt'].dt.day         # 1-31
df2['hour']      = df2['capturedAt'].dt.hour        # 0-23
df2['dayofweek'] = df2['capturedAt'].dt.dayofweek   # 0=Monday, 6=Sunday

print(df2[['capturedAt', 'month', 'day', 'hour', 'dayofweek']].head())

               capturedAt  month  day  hour  dayofweek
0 2025-01-01 22:37:36.424      1    1    22          2
1 2025-01-02 01:45:48.634      1    2     1          3
2 2025-01-02 01:45:48.634      1    2     1          3
3 2025-01-02 01:45:48.634      1    2     1          3
4 2025-01-02 01:45:48.634      1    2     1          3


## 6. Final Check
Ensure all columns are clean before proceeding to feature engineering.

In [8]:
print(f"df2 shape: {df2.shape}")
print(f"\nMissing values:")
missing = df2.isnull().sum()
print(missing[missing > 0] if missing[missing > 0].any() else "No missing values")
print(f"\nData types:")
print(df2.dtypes)

df2 shape: (306226, 29)

Missing values:
No missing values

Data types:
capturedAt                  datetime64[ns]
shopId                               int64
itemId                               int64
modelId                              int64
price                                int64
priceBeforeDiscount                  int64
promotionId                          int64
cat_id                               int64
raw_discount                         int64
show_discount                        int64
brand                               object
is_free_shipping                     int64
is_pre_order                         int64
item_price_min                       int64
item_price_max                       int64
review_rating                      float64
total_rating_count                   int64
cmt_count                            int64
shop_rating                        float64
shop_response_rate                 float64
shop_follower_count                  int64
is_official_shop         

In [9]:
# Save cleaned data for next notebook
df2.to_csv('data_preprocessed.csv', index=False)
print('✓ Saved: data_preprocessed.csv')

✓ Saved: data_preprocessed.csv
